# NuFrost Local Evaluation Notebook

This notebook runs the local accuracy assessment for NuFrost, Zhu2015, and HANTS without Google Colab or Google Drive.

In [ ]:
from pathlib import Path
import os


PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = PROJECT_DIR / "data" / "cache"
DATA_DIR = PROJECT_DIR / "data" / "input"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Info] Project directory: {PROJECT_DIR}")
print(f"[Info] Cache directory: {CACHE_DIR}")
print(f"[Info] Data directory: {DATA_DIR}")

print(f"[Info] Changing working directory to: {PROJECT_DIR}")
os.chdir(str(PROJECT_DIR))

In [ ]:
import importlib
import pandas as pd
from IPython.display import display

import src.evaluation
from config import build_args

importlib.reload(src.evaluation)

In [ ]:
# Modify these parameters manually
IMAGE_NAMES = [
    "COPERNICUS_S2_HARMONIZED_B2_lon91.2734_lat29.7904.tif",
    "COPERNICUS_S2_HARMONIZED_B2_lon91.9113_lat29.1816.tif",
    "COPERNICUS_S2_HARMONIZED_B2_lon94.3610_lat29.6483.tif",
    "COPERNICUS_S2_HARMONIZED_B2_lon92.6035_lat28.9443.tif",
]

print(f"[Info] Number of images to evaluate: {len(IMAGE_NAMES)}")


In [ ]:
print("========== Starting Local Accuracy Assessment ==========")

all_results = []
for image_name in IMAGE_NAMES:
    image_path = DATA_DIR / image_name

    print(f"\n--- Evaluating: {image_name} ---")
    if not image_path.exists():
        print(f"[Warning] File not found: {image_path}. Skipping.")
        continue

    args = build_args({})
    args.image = str(image_path)
    args.cache_dir = str(CACHE_DIR)
    args.n_jobs = -1

    df_results = src.evaluation.evaluate_algorithms(
        image_path=str(args.image),
        args=args,
        num_points=40000,
        n_jobs=args.n_jobs
    )

    df_results.insert(0, "Image", image_name)
    all_results.append(df_results)

if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1200)
    display(final_df)
else:
    print("No valid images processed.")
